# Zero-shot protein binder classification

Does a general-purpose LLM classifier, shown only cofolding metrics, predict whether an
AI-designed binder binds its target in the wet lab?

This notebook reads the scored runs in `results/` and compares them against supervised and
single-feature baselines on the same designs. Produce the inputs first:

```
zsbinders score --label adaptyv --model featherless-ai/Qwen3.8-27B-classifier
zsbinders score --label twist   --model featherless-ai/Qwen3.8-27B-classifier
```

In [1]:
from pathlib import Path

import pandas as pd

from zsbinders import baselines, evaluate, figures

pd.set_option("display.float_format", lambda v: f"{v:.3f}")

runs = {p.stem.replace("scored_", ""): pd.read_csv(p) for p in sorted(Path("../results").glob("scored_*.csv"))}
{name: len(df) for name, df in runs.items()}

{'adaptyv_Qwen3.6-35B-A3B-classifier': 1082,
 'adaptyv_Qwen3.8-27B-classifier': 1082,
 'twist_Qwen3.6-35B-A3B-classifier': 1097,
 'twist_Qwen3.8-27B-classifier': 1097}

## Zero-shot metrics

The fixed 0.5 cut and the threshold that maximises F1 on these same labels. The gap between them is calibration, not ranking: ROC-AUC is identical.

In [2]:
zero_shot = {}
for name, df in runs.items():
    zero_shot.update(evaluate.evaluate_scored(df, name))

evaluate.metrics_frame(zero_shot)

,n,positive_rate,majority_baseline_accuracy,threshold,accuracy,balanced_accuracy,precision,recall,f1,mcc,roc_auc,average_precision
model,,,,,,,,,,,,
adaptyv_Qwen3.6-35B-A3B-classifier @0.5,1082.000,0.272,0.728,0.500,0.331,0.537,0.288,0.990,0.446,0.135,0.694,0.431
adaptyv_Qwen3.6-35B-A3B-classifier @best-f1,1082.000,0.272,0.728,0.999,0.579,0.645,0.371,0.789,0.504,0.260,0.694,0.431
adaptyv_Qwen3.8-27B-classifier @0.5,1082.000,0.272,0.728,0.500,0.272,0.500,0.272,1.000,0.427,0.000,0.687,0.430
adaptyv_Qwen3.8-27B-classifier @best-f1,1082.000,0.272,0.728,0.994,0.660,0.652,0.417,0.636,0.504,0.275,0.687,0.430
twist_Qwen3.6-35B-A3B-classifier @0.5,1097.000,0.259,0.741,0.500,0.298,0.522,0.268,0.986,0.421,0.091,0.649,0.369
twist_Qwen3.6-35B-A3B-classifier @best-f1,1097.000,0.259,0.741,0.993,0.450,0.605,0.311,0.926,0.466,0.219,0.649,0.369
twist_Qwen3.8-27B-classifier @0.5,1097.000,0.259,0.741,0.500,0.259,0.500,0.259,1.000,0.411,0.000,0.648,0.366
twist_Qwen3.8-27B-classifier @best-f1,1097.000,0.259,0.741,0.980,0.485,0.606,0.317,0.856,0.462,0.202,0.648,0.366


## Per-class and per-target breakdown

At 0.5 the classifier calls almost everything a binder, so non-binder recall collapses while binder recall approaches 1.

In [3]:
print(evaluate.report(next(iter(runs.values()))))

threshold: 0.500
confusion matrix [rows=true, cols=pred], order=[non_binder, binder]:
[[ 67 721]
 [  3 291]]

              precision    recall  f1-score   support

  non_binder       0.96      0.09      0.16       788
      binder       0.29      0.99      0.45       294

    accuracy                           0.33      1082
   macro avg       0.62      0.54      0.30      1082
weighted avg       0.78      0.33      0.23      1082

per-target accuracy:
              mean  count
target                   
MBP          0.058     86
TNFa         0.065    123
BBF-14       0.077     78
15-PGDH      0.172     29
EGFR         0.195     82
BHRF1        0.203     59
TrkA         0.250     80
Nipah-G      0.270     63
RBX1         0.308     78
Cas9         0.458     59
PD-L1        0.549     71
IL-7Ra       0.577     78
Latent GDF-8 0.596     52
VEGF-A       0.610     77
TREM2        0.836     67


## Against the baselines

A model trained on the same feature table, and each metric used as a score on its own.

In [4]:
name, df = next(iter(runs.items()))
df = df.dropna(subset=["p_binder"])
y_true = evaluate.binarize(df["y_true"])

candidates = {"zero-shot Jev": df["p_binder"].to_numpy(dtype=float)}
candidates.update(baselines.supervised_scores(df, seed=1337))
candidates.update(baselines.single_feature_scores(df))

rows = {f"{k} @best-f1": evaluate.metrics(y_true, v, evaluate.best_f1_threshold(y_true, v)) for k, v in candidates.items()}
evaluate.metrics_frame(rows).sort_values("roc_auc", ascending=False)

,n,positive_rate,majority_baseline_accuracy,threshold,accuracy,balanced_accuracy,precision,recall,f1,mcc,roc_auc,average_precision
model,,,,,,,,,,,,
logistic regression @best-f1,1082.000,0.272,0.728,0.603,0.809,0.781,0.629,0.721,0.672,0.540,0.861,0.694
gradient boosting @best-f1,1082.000,0.272,0.728,0.210,0.811,0.782,0.633,0.721,0.674,0.544,0.854,0.708
feature: plddt_binder_mean @best-f1,1082.000,0.272,0.728,0.872,0.726,0.716,0.496,0.694,0.579,0.395,0.743,0.481
feature: sc_dockq_mean @best-f1,1082.000,0.272,0.728,0.664,0.634,0.656,0.401,0.704,0.511,0.278,0.705,0.445
feature: ipsae_min_mean @best-f1,1082.000,0.272,0.728,0.728,0.659,0.672,0.423,0.701,0.528,0.308,0.701,0.407
zero-shot Jev @best-f1,1082.000,0.272,0.728,0.999,0.579,0.645,0.371,0.789,0.504,0.260,0.694,0.431
feature: sc_dockq_max @best-f1,1082.000,0.272,0.728,0.689,0.556,0.637,0.360,0.813,0.499,0.250,0.690,0.433
feature: ipsae_min_max @best-f1,1082.000,0.272,0.728,0.767,0.604,0.677,0.392,0.837,0.534,0.318,0.678,0.383
feature: iptm_pae_mean @best-f1,1082.000,0.272,0.728,0.872,0.579,0.645,0.371,0.789,0.505,0.262,0.673,0.382


### Held-out antigens

Stratified folds let a trained model learn each target's base rate. Holding out whole antigens measures what matters: transfer to a target never seen in training.

In [5]:
grouped = {"zero-shot Jev": df["p_binder"].to_numpy(dtype=float)}
grouped.update(baselines.supervised_scores(df, group_by_target=True, seed=1337))

rows = {f"{k} @best-f1": evaluate.metrics(y_true, v, evaluate.best_f1_threshold(y_true, v)) for k, v in grouped.items()}
evaluate.metrics_frame(rows).sort_values("roc_auc", ascending=False)

[warn] single-class targets excluded from grouped CV: ['MBP']


,n,positive_rate,majority_baseline_accuracy,threshold,accuracy,balanced_accuracy,precision,recall,f1,mcc,roc_auc,average_precision
model,,,,,,,,,,,,
zero-shot Jev @best-f1,1082.000,0.272,0.728,0.999,0.579,0.645,0.371,0.789,0.504,0.260,0.694,0.431
logistic regression @best-f1,1082.000,0.272,0.728,0.396,0.589,0.624,0.366,0.701,0.481,0.221,0.613,0.309
gradient boosting @best-f1,1082.000,0.272,0.728,0.002,0.352,0.536,0.288,0.939,0.441,0.101,0.562,0.307


## Figures

In [6]:
out = Path("../figures")
out.mkdir(exist_ok=True)
figures.render_all(runs, out)

[PosixPath('../figures/roc_curves.png'),
 PosixPath('../figures/score_distributions.png'),
 PosixPath('../figures/score_vs_metrics_adaptyv_Qwen3.6-35B-A3B-classifier.png'),
 PosixPath('../figures/score_vs_metrics_adaptyv_Qwen3.8-27B-classifier.png'),
 PosixPath('../figures/score_vs_metrics_twist_Qwen3.6-35B-A3B-classifier.png'),
 PosixPath('../figures/score_vs_metrics_twist_Qwen3.8-27B-classifier.png')]